# Dusk v2: train a Piper voice (Google Colab, free T4)

Dusk is a clean, lively American English voice for Piper, trained on one Hi-Fi TTS reader (CC BY 4.0) and started from the public-domain LJ Speech "high" checkpoint.

How to run it (keyboard friendly):
1. Runtime menu, Change runtime type, choose T4 GPU, Save.
2. Runtime menu, Run all (Ctrl+F9). Allow Google Drive access when asked.
3. Everything is saved in My Drive, folder "Dev/Dusk": data, checkpoints (last 2 only), samples, logs and the finished voice.
4. If Colab disconnects, reconnect and press Run all again. It skips what's done and resumes from the newest checkpoint.

Listen along: every few epochs three test sentences land in Dev/Dusk/samples. The finished voice lands in Dev/Dusk/export as two zips:

- dusk-standard.zip: Dusk Standard, the full-size voice (en_US-dusk-high.onnx + .onnx.json) for NVDA, Android (SherpaTTS), Piper on a computer and Home Assistant.
- dusk-compact-ios.zip: Dusk Compact, a half-size copy plus the two "split" streaming files, for the iPhone app "Piper - Neural TTS".

Dusk Enhanced (the same voice trained longer on more of her clips) is a planned later run: see the commented preset in the Settings cell.

This notebook only reads and writes inside My Drive/Dev/Dusk. It never deletes, lists or changes anything else in your Drive.

In [ ]:
# ===== Settings: the only cell you should need to change =====
DRIVE_FOLDER   = "Dev/Dusk"          # folder in My Drive that keeps all progress (only this folder is touched)
VOICE_KEY      = "en_US-dusk-high"   # file names: en_US-dusk-high.onnx + .onnx.json
SPEAKER_ID     = "11697"             # Hi-Fi TTS reader 11697 (LibriVox reader Celine Major)
HOURS_OF_AUDIO = 3.5                 # training audio to keep (3 to 4 hours)
ANALYZE_HOURS  = 12                  # how much of her 26.6 h to scan for the liveliest clips
MIN_CLIP_SEC, MAX_CLIP_SEC = 1.0, 15.0

MAX_STEPS      = 20000               # total training batches, summed over all sessions
BATCH_SIZE     = 16                  # falls back to 12, then 8, on "CUDA out of memory"
PRECISION      = "32"                # "16-mixed" is faster on a T4 but untested with Piper; go back to "32" if losses show nan
USE_MRD        = True                # Piper's sharper-detail discriminator (training only, free at run time)
SESSION_MAX_HOURS = 11.5             # stop cleanly before Colab's 12 h ceiling
TRAIN          = True                # False = skip training and just export the newest checkpoint

SAMPLE_EVERY_EPOCHS = 10             # render the 3 test sentences to Drive this often
DRIVE_SAVE_MINUTES  = 30             # copy a checkpoint to Drive this often (they are ~1 GB)
KEEP_CHECKPOINTS    = 2              # checkpoints kept on Drive

TARGET_WPM     = 144                 # default speed baked into the .onnx.json
TEST_SENTENCES = [                   # (file name, text, length_scale): 0.5 = about 2x speed
    ("1-normal",   "Hey, I'm Dusk. I can read your books, your messages, and everything in between.", 1.0),
    ("2-fast",     "Messages, three unread. Button. Heading level two, Settings. Link, privacy policy. Double tap to open. Dimmed.", 0.5),
    ("3-question", "Wait, you finished the whole book already? How did it end?", 1.0),
]
EXPORT_SHERPA  = True                # also build a sherpa-onnx package (Android / BrailleNote)
EXPORT_COMPACT = True                # also build Dusk Compact for iPhone: half-size (fp16) copy + split encoder/decoder

# Dusk Enhanced preset (a later, longer run on more of her clips). To use it, remove the # in front of these lines.
# Use a different DRIVE_FOLDER so Enhanced never mixes with the Standard checkpoints.
# DRIVE_FOLDER   = "Dev/Dusk/enhanced"   # stays inside Dev/Dusk
# HOURS_OF_AUDIO = 8                 # more training audio (her full set is 26.6 h)
# ANALYZE_HOURS  = 26.6              # scan all of her clips
# MAX_STEPS      = 60000             # about 3x the training of Standard (more Colab sessions)

# Pinned sources (checked 2026-10-30)
PIPER_REPO, PIPER_COMMIT = "https://github.com/OHF-Voice/piper1-gpl", "74ac24e7e8e2e57a5e1fada5505c74e3d9c1843a"
DATASET_REPO, DATASET_REV = "MikhailT/hifi-tts", "f0a934ddf4e8da0fc6c53fb42fda2c9ac8ed93df"   # parquet copy of openslr.org/109
BASE_REPO, BASE_FILE = "rhasspy/piper-checkpoints", "en/en_US/ljspeech/high/ljspeech-2000.ckpt"

In [ ]:
# Connect Google Drive, make the folders, check the GPU. Safe to re-run.
import os, sys, json, glob, time, shutil, subprocess
from google.colab import drive
if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")

D = f"/content/drive/MyDrive/{DRIVE_FOLDER}"              # everything that must survive a disconnect
LOCAL = "/content/dusk"                                    # fast scratch disk, gone after a disconnect
for d in [f"{D}/{s}" for s in ("data/features", "checkpoints", "samples", "logs", "export")] + [LOCAL]:
    os.makedirs(d, exist_ok=True)
VOICE_CONFIG = f"{D}/data/{VOICE_KEY}.onnx.json"           # written by Piper during training

def log(msg):
    """Print and append to Dusk/logs/notebook.log."""
    line = time.strftime("%Y-%m-%d %H:%M:%S ") + str(msg)
    print(line)
    with open(f"{D}/logs/notebook.log", "a") as f:
        f.write(line + "\n")

def sh(cmd):
    """Run a shell command, show the end of its output, stop on failure."""
    r = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(r.stdout[-2500:])
    if r.returncode:
        raise RuntimeError(f"Command failed ({r.returncode}): {cmd}")

gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout.strip()
if not gpu:   # no GPU (e.g. Colab's daily GPU limit): setup, preview and export still work; the training cell stops
    print("WARNING: No GPU. Training needs Runtime menu > Change runtime type > T4 GPU. Export and preview work without it.")
log(f"Session start. GPU: {gpu}. Drive folder: {D}")

In [ ]:
# Install espeak-ng and Piper's training code (pinned commit). Skips if already installed this session.
REPO = "/content/piper1-gpl"
def piper_ready():
    test = "import piper.espeakbridge, piper.train.vits.monotonic_align, lightning"
    return subprocess.run([sys.executable, "-c", test], cwd="/content", capture_output=True).returncode == 0

if piper_ready():
    log("Piper training code already installed.")
else:
    sh("apt-get -qq update && apt-get -qq install -y espeak-ng build-essential cmake ninja-build > /dev/null")
    if not os.path.isdir(REPO):
        sh(f"git clone -q {PIPER_REPO} {REPO}")
    sh(f"cd {REPO} && git fetch -q origin && git checkout -q {PIPER_COMMIT}")
    sh(f"cd {REPO} && pip install -q -e '.[train]' && pip install -q pytorch-lightning pyarrow soundfile")
    sh(f"cd {REPO} && pip install -q scikit-build && ./build_monotonic_align.sh && python setup.py build_ext --inplace")
    if not piper_ready():
        raise RuntimeError("Piper installed but will not import. See the output above.")
    log("Installed Piper training code.")
sys.path.insert(0, f"{REPO}/src"); sys.path.insert(0, LOCAL)   # so this notebook can import piper + helpers
# Newer PyTorch defaults torch.onnx.export to the dynamo exporter (needs onnxscript, untested with Piper): use the classic one.
_exp = f"{REPO}/src/piper/train/export_onnx.py"
_src = open(_exp).read()
if "dynamo=False" not in _src:
    open(_exp, "w").write(_src.replace("torch.onnx.export(", "torch.onnx.export(dynamo=False, ", 1))

In [ ]:
%%writefile /content/dusk/dusk_audio.py
# Small audio helpers shared by the notebook and the trainer: pitch, noise, loudness, clip reading.
import io, os, zlib
import numpy as np, librosa, soundfile as sf
from scipy.signal import medfilt

FRAME, HOP = 1024, 160   # at 16 kHz: 64 ms window, 10 ms hop

def level_db(y):
    rms = librosa.feature.rms(y=y, frame_length=FRAME, hop_length=HOP)[0]
    return 20 * np.log10(rms + 1e-9)

def pitch_stats(y, sr):
    """Cheap F0 (YIN) -> pitch spread in semitones (std), median, voiced share."""
    if sr != 16000:
        y = librosa.resample(y, orig_sr=sr, target_sr=16000, res_type="soxr_hq")
    f0 = librosa.yin(y, fmin=70, fmax=450, sr=16000, frame_length=FRAME, hop_length=HOP)
    db, zcr = level_db(y), librosa.feature.zero_crossing_rate(y, frame_length=FRAME, hop_length=HOP)[0]
    n = min(len(f0), len(db), len(zcr)); f0, db, zcr = f0[:n], db[:n], zcr[:n]
    st = 12 * np.log2(f0 / 100.0)
    good = (db > np.percentile(db, 95) - 25) & (zcr < 0.12) & (np.abs(st - medfilt(st, 5)) < 1.0)
    if good.sum() < 20:
        return {"pitch_std": np.nan, "pitch_med": np.nan, "voiced": float(good.mean())}
    return {"pitch_std": float(np.std(st[good])), "pitch_med": float(np.median(st[good])), "voiced": float(good.mean())}

def noise_stats(y16):
    db = level_db(y16)
    return {"floor_db": float(np.percentile(db, 10)), "speech_db": float(np.percentile(db, 95))}

def normalize(y, target_db=-20.0, peak=0.89):
    """Speech frames to about -20 dBFS, peaks under -1 dBFS."""
    db = level_db(y); loud = db > np.percentile(db, 95) - 30
    level = 10 * np.log10(np.mean(10 ** (db[loud] / 10)) + 1e-12)
    y = y * 10 ** ((target_db - level) / 20)
    m = np.abs(y).max()
    return y * (peak / m) if m > peak else y

def clip_id(path):   # audio/11697_other/15543/book_04_x_0056.flac -> 15543_book_04_x_0056
    parts = path.split("/")
    return f"{parts[-2]}_{os.path.splitext(parts[-1])[0]}"

def clean_text(t):
    for a, b in (("\u2019", "'"), ("\u2018", "'"), ("\u201c", '"'), ("\u201d", '"'), ("\u2014", ", "), ("|", " ")):
        t = t.replace(a, b)
    return " ".join(t.split())

def _rows(path, rg, speaker, lo, hi, keep_frac):
    import pyarrow.parquet as pq
    pf = pq.ParquetFile(path)
    meta = pf.read_row_group(rg, columns=["speaker", "file", "duration", "text_normalized"]).to_pylist()
    want = [i for i, r in enumerate(meta) if r["speaker"] == speaker and lo <= r["duration"] <= hi
            and zlib.crc32(r["file"].encode()) % 10000 < keep_frac * 10000]
    audio = pf.read_row_group(rg, columns=["audio"]).column("audio") if want else None
    for i in want:
        y, sr = sf.read(io.BytesIO(audio[i].as_py()["bytes"]), dtype="float32")
        yield meta[i], (y.mean(1) if y.ndim > 1 else y), sr

def analyze_row_group(path, rg, speaker, lo, hi, keep_frac):
    """Features for one parquet row group: pitch spread, noise floor, words per minute."""
    out = []
    for r, y, sr in _rows(path, rg, speaker, lo, hi, keep_frac):
        y16 = librosa.resample(y, orig_sr=sr, target_sr=16000, res_type="soxr_hq")
        out.append({"id": clip_id(r["file"]), "shard": os.path.basename(path), "rg": rg,
                    "duration": r["duration"], "text": clean_text(r["text_normalized"]),
                    "wpm": len(r["text_normalized"].split()) / r["duration"] * 60,
                    **pitch_stats(y16, 16000), **noise_stats(y16)})
    return out

def write_row_group(path, rg, speaker, ids, out_dir, sr_out=22050):
    """Write the chosen clips of one row group as 22.05 kHz mono 16-bit wavs."""
    done = []
    for r, y, sr in _rows(path, rg, speaker, 0, 1e9, 1.0):
        cid = clip_id(r["file"])
        if cid in ids:
            y = normalize(librosa.resample(y, orig_sr=sr, target_sr=sr_out, res_type="soxr_hq"))
            sf.write(os.path.join(out_dir, cid + ".wav"), y, sr_out, subtype="PCM_16")
            done.append(cid)
    return done

In [ ]:
# Find which parquet files hold reader 11697, and download only those (about 3.6 GB, scratch disk).
import importlib, dusk_audio; importlib.reload(dusk_audio)
from huggingface_hub import HfFileSystem, hf_hub_download
DATA_DONE = os.path.exists(f"{D}/data/metadata.csv") and os.path.exists(f"{D}/data/wavs.tar")
SHARDS_JSON = f"{D}/data/shards.json"

def row_groups_with_speaker(fname):
    import pyarrow.parquet as pq
    fs = HfFileSystem()
    with fs.open(f"datasets/{DATASET_REPO}@{DATASET_REV}/{fname}", "rb", block_size=1 << 16) as f:
        md_ = pq.ParquetFile(f).metadata
        col = md_.schema.to_arrow_schema().get_field_index("speaker")
        return [g for g in range(md_.num_row_groups)
                if (s := md_.row_group(g).column(col).statistics) is None or s.min <= SPEAKER_ID <= s.max]

if DATA_DONE:
    log("Training data already on Drive, skipping download.")
else:
    if not os.path.exists(SHARDS_JSON):
        from concurrent.futures import ThreadPoolExecutor
        files = [f for f in HfFileSystem().ls(f"datasets/{DATASET_REPO}@{DATASET_REV}/data", detail=False)
                 if os.path.basename(f).startswith("train.")]
        names = ["data/" + os.path.basename(f) for f in files]
        with ThreadPoolExecutor(16) as ex:
            plan = {n: g for n, g in zip(names, ex.map(row_groups_with_speaker, names)) if g}
        json.dump(plan, open(SHARDS_JSON, "w"), indent=1)
    SHARDS = json.load(open(SHARDS_JSON))
    log(f"Reader {SPEAKER_ID} is in {len(SHARDS)} parquet files.")
    LOCAL_SHARD = {n: hf_hub_download(DATASET_REPO, n, repo_type="dataset", revision=DATASET_REV,
                                      local_dir=f"{LOCAL}/hf") for n in SHARDS}
    log("Parquet files ready on scratch disk.")

In [ ]:
# Measure pitch spread, noise floor and pace for a sample of her clips. One CSV per file on Drive, so it resumes.
from concurrent.futures import ProcessPoolExecutor
SPEAKER_HOURS = 26.6                       # her total in Hi-Fi TTS (measured 2026-10-30); ~28x real time per CPU core
keep_frac = min(1.0, ANALYZE_HOURS / SPEAKER_HOURS)

if not DATA_DONE:
    import pandas as pd
    for name, groups in SHARDS.items():
        out_csv = f"{D}/data/features/{os.path.basename(name)}.csv"
        if os.path.exists(out_csv):
            continue
        t0 = time.time()
        with ProcessPoolExecutor(os.cpu_count()) as ex:
            jobs = [ex.submit(dusk_audio.analyze_row_group, LOCAL_SHARD[name], g, SPEAKER_ID,
                              MIN_CLIP_SEC, MAX_CLIP_SEC, keep_frac) for g in groups]
            rows = [r for j in jobs for r in j.result()]
        pd.DataFrame(rows).to_csv(out_csv, index=False)
        log(f"Analyzed {len(rows)} clips in {os.path.basename(name)} ({time.time() - t0:.0f} s).")

In [ ]:
# Pick the liveliest clean clips: drop the noisiest 20%, odd character voices and tracking errors,
# then rank by pitch spread (favoring her typical pace) until HOURS_OF_AUDIO is filled.
import pandas as pd
SEL_CSV = f"{D}/data/selection.csv"
if not DATA_DONE and not os.path.exists(SEL_CSV):
    df = pd.concat(pd.read_csv(f) for f in glob.glob(f"{D}/data/features/*.csv")).dropna(subset=["pitch_std"])
    pool = df.copy()
    df = df[df.text.str.fullmatch(r"[A-Za-z ,.;:!?'\"()\-]+") & (df.voiced >= 0.25) & (df.pitch_std <= 7)]
    df = df[df.floor_db <= df.floor_db.quantile(0.8)]
    df = df[(df.pitch_med - df.pitch_med.median()).abs() <= 3]
    z = lambda s: (s - s.mean()) / s.std()
    df["score"] = z(df.pitch_std) - 0.3 * z((df.wpm - df.wpm.median()).abs())   # lively pitch, typical pace
    df = df.sort_values("score", ascending=False)
    sel = df[df.duration.cumsum() <= HOURS_OF_AUDIO * 3600]
    sel.to_csv(SEL_CSV, index=False)
    stats = {"clips": len(sel), "hours": round(sel.duration.sum() / 3600, 2),
             "pitch_std_selected": round(sel.pitch_std.median(), 2), "pitch_std_pool": round(pool.pitch_std.median(), 2),
             "wpm_selected": round(sel.wpm.median(), 1), "wpm_pool": round(pool.wpm.median(), 1),
             "floor_db_selected": round(sel.floor_db.median(), 1)}
    json.dump(stats, open(f"{D}/data/stats.json", "w"), indent=1)
    log(f"Selected: {stats}  (target pitch spread 3.5-3.8 semitones)")

In [ ]:
# Write the chosen clips as 22.05 kHz wavs + LJSpeech-style metadata.csv, pack them to Drive.
# Next sessions just unpack them from Drive.
WAVS = f"{LOCAL}/wavs"
if not DATA_DONE:
    sel = pd.read_csv(SEL_CSV)
    os.makedirs(WAVS, exist_ok=True)
    with ProcessPoolExecutor(os.cpu_count()) as ex:
        jobs = [ex.submit(dusk_audio.write_row_group, LOCAL_SHARD[n], int(g), SPEAKER_ID,
                          set(sel[(sel.shard == os.path.basename(n)) & (sel.rg == g)]["id"]), WAVS)
                for n in SHARDS for g in sorted(sel[sel.shard == os.path.basename(n)].rg.unique())]
        written = {c for j in jobs for c in j.result()}
    sel = sel[sel["id"].isin(written)]
    with open(f"{LOCAL}/metadata.csv", "w", encoding="utf-8") as f:
        f.writelines(f"{r.id}.wav|{r.text}\n" for r in sel.itertuples())
    sh(f"tar -cf {LOCAL}/wavs.tar -C {LOCAL} wavs && cp {LOCAL}/wavs.tar {D}/data/wavs.tar.part "
       f"&& mv {D}/data/wavs.tar.part {D}/data/wavs.tar && cp {LOCAL}/metadata.csv {D}/data/metadata.csv")
    shutil.rmtree(f"{LOCAL}/hf", ignore_errors=True); os.remove(f"{LOCAL}/wavs.tar")
    log(f"Dataset saved to Drive: {len(sel)} clips.")
elif not os.path.isdir(WAVS):
    sh(f"tar -xf {D}/data/wavs.tar -C {LOCAL} && cp {D}/data/metadata.csv {LOCAL}/metadata.csv")
    log("Dataset unpacked from Drive.")
N_CLIPS = sum(1 for _ in open(f"{LOCAL}/metadata.csv"))
print(N_CLIPS, "clips ready")

In [ ]:
# Where to start: the newest Dusk checkpoint (resume), else the LJ Speech high checkpoint (fresh start).
def newest_ckpt():
    """Newest Dusk checkpoint on scratch disk or Drive (names sort by epoch), or None."""
    found = glob.glob(f"{LOCAL}/ckpt/dusk-ckpt-*.ckpt") + glob.glob(f"{D}/checkpoints/dusk-ckpt-*.ckpt")
    return max(found, key=os.path.basename) if found else None
for f in glob.glob(f"{D}/checkpoints/*.part"):
    os.remove(f)                                                       # half-copied file from a dropped session
RESUME = newest_ckpt()
BASE_SLIM = f"{LOCAL}/base/ljspeech-2000-weights.ckpt"

if RESUME:
    log(f"Will resume from {os.path.basename(RESUME)}")
elif not os.path.exists(BASE_SLIM):
    import torch
    p = hf_hub_download(BASE_REPO, BASE_FILE, repo_type="dataset", local_dir=f"{LOCAL}/base")
    ck = torch.load(p, map_location="cpu", weights_only=False)        # old checkpoint: keep weights only
    torch.save({"state_dict": ck["state_dict"]}, BASE_SLIM)
    del ck; os.remove(p)
    log("Base checkpoint ready (LJ Speech high, public domain).")

In [ ]:
# Dusk Preview 1: an early, importable test voice from the newest checkpoint (added 2026-10-30).
# Distinct identity (dataset "dusk_preview_1") so the iPhone app treats it as a NEW voice. Saves to Dev/Dusk/export/preview-1.
# Also fixes train_plan.json batch 16 -> 12 (16 runs out of GPU memory on a T4). Set MAKE_PREVIEW = False to skip.
MAKE_PREVIEW = not os.path.exists(f"{D}/export/preview-1/dusk-preview-1.onnx")
_plan = f"{D}/data/train_plan.json"
if os.path.exists(_plan):
    _p = json.load(open(_plan))
    if _p.get("batch_size") != 12:
        _p["batch_size"] = 12; json.dump(_p, open(_plan, "w")); log(f"train_plan.json now {_p}")
if MAKE_PREVIEW:
    import wave, warnings
    sh("pip install -q onnxconverter-common==1.16.0")
    import onnx, numpy as np
    from onnx import TensorProto, numpy_helper
    from onnxconverter_common import float16
    PV_KEY = "dusk-preview-1"
    pv_ck = newest_ckpt()
    assert pv_ck, "No Dusk checkpoint yet."
    PV = f"{LOCAL}/preview-1"; PV_DRIVE = f"{D}/export/preview-1"
    os.makedirs(PV, exist_ok=True); os.makedirs(PV_DRIVE, exist_ok=True)
    pv32, pv16, pvj = f"{PV}/{PV_KEY}-fp32.onnx", f"{PV}/{PV_KEY}.onnx", f"{PV}/{PV_KEY}.onnx.json"
    sh(f'cd {LOCAL} && {sys.executable} -m piper.train.export_onnx --checkpoint "{pv_ck}" --output-file "{pv32}"')
    pcfg = json.load(open(VOICE_CONFIG))
    pcfg.update(dataset="dusk_preview_1", name="Dusk Preview 1",
                language={"code": "en_US", "family": "en", "region": "US", "name_native": "English",
                          "name_english": "English", "country_english": "United States"})
    pcfg["audio"]["quality"] = "high"
    pcfg["inference"] = {"noise_scale": 0.667, "length_scale": 1.0, "noise_w": 0.8}
    json.dump(pcfg, open(pvj, "w"), indent=2, ensure_ascii=False)

    def _to_fp16(model):
        m16 = float16.convert_float_to_float16(model, keep_io_types=True)
        types = {v.name: v.type.tensor_type.elem_type for v in list(m16.graph.value_info) + list(m16.graph.output)}
        n_fixed = 0
        for n in m16.graph.node:
            if not n.output or types.get(n.output[0]) != TensorProto.FLOAT16:
                continue
            for a in n.attribute:
                if n.op_type == "Cast" and a.name == "to" and a.i == TensorProto.FLOAT:
                    a.i = TensorProto.FLOAT16; n_fixed += 1
                elif n.op_type in ("RandomNormalLike", "RandomUniformLike", "RandomNormal", "RandomUniform", "EyeLike") \
                        and a.name == "dtype" and a.i == TensorProto.FLOAT:
                    a.i = TensorProto.FLOAT16; n_fixed += 1
                elif n.op_type == "ConstantOfShape" and a.name == "value" and a.t.data_type == TensorProto.FLOAT:
                    a.t.CopyFrom(numpy_helper.from_array(numpy_helper.to_array(a.t).astype(np.float16), a.t.name)); n_fixed += 1
        return m16, n_fixed
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        m16, n_fixed = _to_fp16(onnx.load(pv32))
    onnx.save(m16, pv16)

    from piper import PiperVoice, SynthesisConfig
    def pv_render(text, path, ls=None):
        with wave.open(path, "wb") as w:
            PiperVoice.load(pv16, config_path=pvj).synthesize_wav(text, w, syn_config=SynthesisConfig(length_scale=ls))
        with wave.open(path) as w:
            return w.getnframes() / w.getframerate()
    para = ("Here is a short paragraph for timing. It has a few sentences, some commas, and a question. "
            "Does it sound natural? I hope it does, because this is the speed you will hear by default.")
    wpm = len(para.split()) / pv_render(para, f"{PV}/timing.wav", 1.0) * 60
    pcfg["inference"]["length_scale"] = round(min(1.5, max(0.75, wpm / TARGET_WPM)), 3)
    json.dump(pcfg, open(pvj, "w"), indent=2, ensure_ascii=False)
    os.remove(f"{PV}/timing.wav")
    pv_render(TEST_SENTENCES[0][1], f"{PV}/{PV_KEY}-sample-1.wav")
    pv_render(TEST_SENTENCES[2][1], f"{PV}/{PV_KEY}-sample-2.wav")
    for f in (pv16, pvj, f"{PV}/{PV_KEY}-sample-1.wav", f"{PV}/{PV_KEY}-sample-2.wav"):
        shutil.copy(f, f"{PV_DRIVE}/{os.path.basename(f)}")
    mb = lambda p: os.path.getsize(p) / 1e6
    log(f"[preview] Dusk Preview 1 from {os.path.basename(pv_ck)}: fp16 {mb(pv16):.1f} MB (fp32 {mb(pv32):.1f} MB), "
        f"json {os.path.getsize(pvj)} bytes, {n_fixed} type fixes, {wpm:.0f} wpm at 1.0, length_scale {pcfg['inference']['length_scale']}, "
        f"sample_rate {pcfg['audio']['sample_rate']}, phonemes {len(pcfg.get('phoneme_id_map', {}))}. Saved to {PV_DRIVE}")
    del m16

In [ ]:
%%writefile /content/dusk/train_dusk.py
# Piper's trainer with Drive-friendly extras: epoch checkpoints, a Drive copy every N minutes
# (keep last K), test sentences rendered to Drive, and plain-text progress lines.
import glob, json, os, shutil, sys, threading, time
import numpy as np, torch
from lightning.pytorch.callbacks import Callback, ModelCheckpoint
import piper.train.__main__ as piper_main

C = json.loads(os.environ["DUSK_CFG"])
sys.path.insert(0, C["local"])
import dusk_audio

class DriveSync(Callback):
    """Copies the newest local checkpoint to Drive in the background; keeps the last K there."""
    def __init__(self):
        self.last, self.thread = time.time(), None
    def _copy(self, link, name):
        dst = os.path.join(C["drive_ckpt"], name)
        try:
            shutil.copyfile(link, dst + ".part"); os.replace(dst + ".part", dst)
            for old in sorted(glob.glob(os.path.join(C["drive_ckpt"], "dusk-ckpt-*.ckpt")))[:-C["keep"]]:
                os.remove(old)
            print(f"[dusk] checkpoint saved to Drive: {name}", flush=True)
        except Exception as e:
            print(f"[dusk] Drive copy failed, will try again later: {e}", flush=True)
        finally:
            os.remove(link)
    def sync(self, force=False):
        if self.thread and self.thread.is_alive():
            if not force:
                return
            self.thread.join()
        files = sorted(glob.glob(os.path.join(C["local_ckpt"], "dusk-ckpt-*.ckpt")))
        if not files or os.path.exists(os.path.join(C["drive_ckpt"], os.path.basename(files[-1]))):
            return
        link = files[-1] + ".sync"                       # hard link: survives the next epoch's cleanup
        if os.path.exists(link):
            os.remove(link)
        os.link(files[-1], link)
        self.last = time.time()
        self.thread = threading.Thread(target=self._copy, args=(link, os.path.basename(files[-1])))
        self.thread.start()
        if force:
            self.thread.join()
    def on_train_epoch_start(self, trainer, pl_module):
        if time.time() - self.last >= C["save_minutes"] * 60:
            self.sync()
    def on_train_end(self, trainer, pl_module):
        self.sync(force=True)
    def on_exception(self, trainer, pl_module, exception):
        self.sync(force=True)

class Samples(Callback):
    """Renders the test sentences with the model in memory; writes wavs + pitch/pace numbers to Drive."""
    def __init__(self):
        self.first, self.phonemizer = True, None
    def on_train_epoch_start(self, trainer, pl_module):
        e = trainer.current_epoch
        if self.first or e % C["sample_every"] == 0:
            self.first = False
            try:
                self.render(trainer, pl_module)
            except Exception as ex:
                print(f"[dusk] sample render skipped: {ex}", flush=True)
    def render(self, trainer, pl_module):
        from piper.config import PiperConfig
        from piper.phoneme_ids import phonemes_to_ids
        from piper.phonemize_espeak import EspeakPhonemizer
        vc = PiperConfig.from_dict(json.load(open(C["voice_config"])))
        self.phonemizer = self.phonemizer or EspeakPhonemizer()
        out = os.path.join(C["drive_samples"], f"epoch-{trainer.current_epoch:04d}")
        os.makedirs(out, exist_ok=True)
        pl_module.eval(); lines = []
        try:
            with torch.no_grad():
                for name, text, ls in C["sentences"]:
                    parts = []
                    for ph in self.phonemizer.phonemize(vc.espeak_voice, text, vowel_clusters=vc.vowel_clusters):
                        if ph:
                            ids = torch.LongTensor(phonemes_to_ids(ph, vc.phoneme_id_map)).unsqueeze(0).to(pl_module.device)
                            lens = torch.LongTensor([ids.shape[1]]).to(pl_module.device)
                            parts.append(pl_module(ids, lens, [0.667, ls, 0.8]).float().cpu().numpy().reshape(-1))
                    y = np.concatenate(parts); y = 0.9 * y / max(0.01, np.abs(y).max())
                    import soundfile as sf
                    sf.write(os.path.join(out, name + ".wav"), y, vc.sample_rate, subtype="PCM_16")
                    p = dusk_audio.pitch_stats(y, vc.sample_rate)
                    wpm = len(text.split()) / (len(y) / vc.sample_rate) * 60
                    lines.append(f"{trainer.current_epoch},{name},{wpm:.0f},{p['pitch_std']:.2f}\n")
        finally:
            pl_module.train()
        with open(os.path.join(C["drive_samples"], "metrics.csv"), "a") as f:
            f.writelines(lines)
        print(f"[dusk] samples for epoch {trainer.current_epoch} saved (epoch,name,wpm,pitch_std): {''.join(lines)}", flush=True)

class Progress(Callback):
    """A short readable line every N batches instead of a progress bar."""
    def on_train_start(self, trainer, pl_module):
        self.t0, self.e0 = time.time(), trainer.current_epoch
    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx):
        if batch_idx % C["print_every"]:
            return
        m, nb = trainer.callback_metrics, max(1, trainer.num_training_batches)
        done = trainer.current_epoch - self.e0 + (batch_idx + 1) / nb
        left_h = (trainer.max_epochs - trainer.current_epoch - (batch_idx + 1) / nb) * (time.time() - self.t0) / max(done, 1e-3) / 3600
        g, d = (float(m.get(k, float("nan"))) for k in ("loss_g", "loss_d"))
        print(f"[dusk] epoch {trainer.current_epoch + 1}/{trainer.max_epochs} batch {batch_idx + 1}/{nb} "
              f"loss_g {g:.2f} loss_d {d:.2f}, about {left_h:.1f} h of training left", flush=True)

piper_main._DEFAULT_CALLBACKS = [
    ModelCheckpoint(dirpath=C["local_ckpt"], filename="dusk-ckpt-e{epoch:04d}-s{step:08d}",
                    auto_insert_metric_name=False, monitor=None, save_top_k=1, every_n_epochs=1,
                    save_on_train_epoch_end=True),
    DriveSync(), Samples(), Progress(),
]
if __name__ == "__main__":
    piper_main.main()

In [ ]:
# Train (or resume). Output streams here and into Dusk/logs. Re-running continues from the newest checkpoint.
import math
PLAN = f"{D}/data/train_plan.json"
if not os.path.exists(PLAN):                                       # fixed once, so the LR schedule stays the same
    per_epoch = math.ceil(N_CLIPS * 0.95 / BATCH_SIZE)
    json.dump({"max_epochs": math.ceil(MAX_STEPS / per_epoch), "batch_size": BATCH_SIZE}, open(PLAN, "w"))
plan = json.load(open(PLAN))
HIGH = ["--model.resblock", "1", "--model.resblock_kernel_sizes", "[3,7,11]",
        "--model.resblock_dilation_sizes", "[[1,3,5],[1,3,5],[1,3,5]]", "--model.upsample_rates", "[8,8,2,2]",
        "--model.upsample_initial_channel", "512", "--model.upsample_kernel_sizes", "[16,16,4,4]"]
os.environ["DUSK_CFG"] = json.dumps({
    "local": LOCAL, "local_ckpt": f"{LOCAL}/ckpt", "drive_ckpt": f"{D}/checkpoints", "keep": KEEP_CHECKPOINTS,
    "save_minutes": DRIVE_SAVE_MINUTES, "drive_samples": f"{D}/samples", "sample_every": SAMPLE_EVERY_EPOCHS,
    "sentences": TEST_SENTENCES, "voice_config": VOICE_CONFIG, "print_every": 50})

def train_once(batch):
    resume = newest_ckpt()
    cmd = [sys.executable, f"{LOCAL}/train_dusk.py", "fit", "--seed_everything", "1234",
           "--data.voice_name", "dusk", "--data.csv_path", f"{LOCAL}/metadata.csv", "--data.audio_dir", WAVS,
           "--data.cache_dir", f"{LOCAL}/cache", "--data.config_path", VOICE_CONFIG, "--data.espeak_voice", "en-us",
           "--data.batch_size", str(batch), "--data.validation_split", "0.05", "--data.num_test_examples", "3",
           "--data.num_workers", "2", "--model.sample_rate", "22050", *HIGH, "--model.use_mrd", str(USE_MRD).lower(),
           "--trainer.accelerator", "gpu", "--trainer.devices", "1", "--trainer.precision", PRECISION,
           "--trainer.max_epochs", str(plan["max_epochs"]), "--trainer.max_time", f"00:{int(SESSION_MAX_HOURS):02d}:{int(SESSION_MAX_HOURS % 1 * 60):02d}:00",
           "--trainer.check_val_every_n_epoch", "5", "--trainer.log_every_n_steps", "50",
           "--trainer.enable_progress_bar", "false", "--trainer.default_root_dir", f"{LOCAL}/run"]
    cmd += ["--ckpt_path", resume] if resume else ["--model.warmstart_ckpt", BASE_SLIM]
    log(f"Training: batch {batch}, {plan['max_epochs']} epochs total, start: {os.path.basename(resume or BASE_SLIM)}")
    tail = []
    with subprocess.Popen(cmd, cwd=LOCAL, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT) as p, \
         open(f"{D}/logs/train-{time.strftime('%Y%m%d-%H%M')}.log", "a") as lf:
        for line in p.stdout:
            lf.write(line)
            tail = (tail + [line])[-40:]
            if line.startswith("[dusk]") or "warmstart" in line or "Error" in line or "Traceback" in line:
                print(line, end="")
    return p.returncode, "".join(tail)

if TRAIN and not gpu:
    raise RuntimeError("No GPU. Runtime menu > Change runtime type > T4 GPU > Save, then Run all again.")
if TRAIN:
    for batch in sorted({b for b in (plan["batch_size"], 12, 8) if b <= plan["batch_size"]}, reverse=True):
        code_, tail = train_once(batch)
        if code_ == 0 or "out of memory" not in tail.lower():
            break
        log(f"Out of GPU memory at batch {batch}, trying smaller.")
    print(tail[-2000:])
    log("Training finished for this session." if code_ == 0 else "Training stopped with an error (see above).")

In [ ]:
# Dusk Standard: export the newest checkpoint as the full-size (fp32) en_US-dusk-high.onnx + .onnx.json,
# with a default speed set to hit TARGET_WPM. This is the version for NVDA, Android and Piper on a computer.
import wave
newest = newest_ckpt()
if not newest:
    raise RuntimeError("No Dusk checkpoint yet. Train first.")
EXP = f"{LOCAL}/export"; os.makedirs(EXP, exist_ok=True)
ONNX, JSON_ = f"{EXP}/{VOICE_KEY}.onnx", f"{EXP}/{VOICE_KEY}.onnx.json"
sh(f'cd {LOCAL} && {sys.executable} -m piper.train.export_onnx --checkpoint "{newest}" --output-file "{ONNX}"')

cfg = json.load(open(VOICE_CONFIG))
cfg.update(dataset="dusk", language={"code": "en_US", "family": "en", "region": "US", "name_native": "English",
                                     "name_english": "English", "country_english": "United States"})
cfg["audio"]["quality"] = "high"
cfg["inference"] = {"noise_scale": 0.667, "length_scale": 1.0, "noise_w": 0.8}
assert cfg["dataset"] == "dusk" and cfg["audio"]["quality"] == "high"   # the voice is called "Dusk" in every app
json.dump(cfg, open(JSON_, "w"), indent=2, ensure_ascii=False)

from piper import PiperVoice, SynthesisConfig
def render(text, path, ls=None):
    with wave.open(path, "wb") as w:
        PiperVoice.load(ONNX, config_path=JSON_).synthesize_wav(text, w, syn_config=SynthesisConfig(length_scale=ls))
    with wave.open(path) as w:
        return w.getnframes() / w.getframerate()
para = ("Here is a short paragraph for timing. It has a few sentences, some commas, and a question. "
        "Does it sound natural? I hope it does, because this is the speed you will hear by default.")
wpm = len(para.split()) / render(para, f"{EXP}/timing.wav", 1.0) * 60
cfg["inference"]["length_scale"] = round(min(1.5, max(0.75, wpm / TARGET_WPM)), 3)
json.dump(cfg, open(JSON_, "w"), indent=2, ensure_ascii=False)
for name, text, ls in TEST_SENTENCES:
    render(text, f"{EXP}/sample-{name}.wav", None if ls == 1.0 else ls * cfg["inference"]["length_scale"])
os.remove(f"{EXP}/timing.wav")
log(f"Exported from {os.path.basename(newest)}: {wpm:.0f} wpm at length_scale 1.0, "
    f"default length_scale set to {cfg['inference']['length_scale']} for about {TARGET_WPM} wpm.")

## Dusk Compact (iPhone): how it's made, and what's been checked

The iPhone app "Piper - Neural TTS" runs voices inside a small background part of the app that iOS stops at about 250 MB of memory. A full-size "high" voice is about 110 MB, and when iOS stops it on long text, VoiceOver quietly switches back to its default voice. The app's own voices avoid this with a half-size copy, so Dusk Compact does the same:

1. `en_US-dusk-high.onnx` (Compact): a half-precision (fp16) copy of the voice, about 58 MB. It's made with onnxconverter-common's float16 converter (inputs and outputs stay 32-bit), plus a small fix so a few type-carrying nodes (Cast, RandomNormalLike) agree with the converted graph. Without that fix the converted file doesn't load.
2. `en_US-dusk-high.encoder.onnx` and `en_US-dusk-high.decoder.onnx`: the Compact voice split into two parts with Piper's own `split_voice_for_streaming` (in the pinned Piper commit). The app streams these, so speech starts sooner and peak memory is lower.
3. `en_US-dusk-high.onnx.json`: the same settings file as Standard (name "dusk", quality "high").

Checked (2026-10-30, on a computer, using the public LJ Speech "high" voice as a stand-in because Dusk isn't trained yet): this exact recipe made files that match the app's own published LJ Speech Compact files. Same inputs and outputs, sizes within 2 KB, and identical audio with randomness turned off, for both the half-size file and the split pair.

UNVERIFIED:
- Nobody has imported Dusk Compact on an iPhone yet. That happens once Dusk is trained.
- Split voices are new in the app (the 1.0.16 test build). The App Store version may not accept the two split files yet. The half-size .onnx + .onnx.json work on their own without them.
- The next cell renders the same sentence from the full-size voice, the half-size voice and the split pair, so you can listen and compare.

In [ ]:
# Dusk Compact (iPhone): half-size fp16 copy + split encoder/decoder, then render the same sentence from each to compare.
if EXPORT_COMPACT:
    sh("pip install -q onnxconverter-common==1.16.0")
    import onnx, numpy as np, onnxruntime
    from onnx import TensorProto, numpy_helper
    from onnxconverter_common import float16
    from piper.split_voice_for_streaming import split_model

    COMPACT = f"{EXP}/compact-ios"; os.makedirs(COMPACT, exist_ok=True)
    C_ONNX, C_JSON = f"{COMPACT}/{VOICE_KEY}.onnx", f"{COMPACT}/{VOICE_KEY}.onnx.json"
    C_ENC, C_DEC = f"{COMPACT}/{VOICE_KEY}.encoder.onnx", f"{COMPACT}/{VOICE_KEY}.decoder.onnx"

    def to_fp16(model):
        """fp16 copy with fp32 inputs/outputs. Then make type-carrying nodes agree with the types the converter gave
        their outputs; without this the file fails to load ("Type Error ... Cast ... does not match")."""
        m16 = float16.convert_float_to_float16(model, keep_io_types=True)
        types = {v.name: v.type.tensor_type.elem_type for v in list(m16.graph.value_info) + list(m16.graph.output)}
        fixed = []
        for n in m16.graph.node:
            if not n.output or types.get(n.output[0]) != TensorProto.FLOAT16:
                continue
            for a in n.attribute:
                if n.op_type == "Cast" and a.name == "to" and a.i == TensorProto.FLOAT:
                    a.i = TensorProto.FLOAT16; fixed.append(n.name)
                elif n.op_type in ("RandomNormalLike", "RandomUniformLike", "RandomNormal", "RandomUniform", "EyeLike") \
                        and a.name == "dtype" and a.i == TensorProto.FLOAT:
                    a.i = TensorProto.FLOAT16; fixed.append(n.name)
                elif n.op_type == "ConstantOfShape" and a.name == "value" and a.t.data_type == TensorProto.FLOAT:
                    a.t.CopyFrom(numpy_helper.from_array(numpy_helper.to_array(a.t).astype(np.float16), a.t.name))
                    fixed.append(n.name)
        return m16, fixed

    import warnings
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")                       # the converter warns about tiny weights rounding; harmless
        m16, fixed = to_fp16(onnx.load(ONNX))
    onnx.save(m16, C_ONNX)
    encoder, decoder = split_model(onnx.load(C_ONNX))         # Piper's own splitter (pinned commit); latent tensor "z"
    onnx.save(encoder, C_ENC); onnx.save(decoder, C_DEC)
    shutil.copy(JSON_, C_JSON)                                # same settings file: name "dusk", quality "high"
    assert json.load(open(C_JSON))["dataset"] == "dusk" and json.load(open(C_JSON))["audio"]["quality"] == "high"
    mb = lambda p: os.path.getsize(p) / 1e6
    log(f"Compact: fp16 {mb(C_ONNX):.1f} MB (full {mb(ONNX):.1f} MB), encoder {mb(C_ENC):.1f} MB, "
        f"decoder {mb(C_DEC):.1f} MB; type fixes on {len(fixed)} nodes.")

    # Same sentence from full-size, Compact and the split pair, saved next to each other for listening.
    name, text, _ = TEST_SENTENCES[0]
    def render_with(model_path, path):
        with wave.open(path, "wb") as w:
            PiperVoice.load(model_path, config_path=JSON_).synthesize_wav(text, w)
    render_with(ONNX, f"{EXP}/compare-{name}-standard-fp32.wav")
    render_with(C_ONNX, f"{EXP}/compare-{name}-compact-fp16.wav")

    voice = PiperVoice.load(C_ONNX, config_path=JSON_)
    ids = [i for sent in voice.phonemize(text) for i in voice.phonemes_to_ids(sent)]
    feed = lambda scales: {"input": np.array([ids], dtype=np.int64), "input_lengths": np.array([len(ids)], dtype=np.int64),
                           "scales": np.array(scales, dtype=np.float32)}
    enc_s = onnxruntime.InferenceSession(C_ENC, providers=["CPUExecutionProvider"])
    dec_s = onnxruntime.InferenceSession(C_DEC, providers=["CPUExecutionProvider"])
    full_s = onnxruntime.InferenceSession(C_ONNX, providers=["CPUExecutionProvider"])
    inf = cfg["inference"]
    audio = dec_s.run(None, {"z": enc_s.run(None, feed([inf["noise_scale"], inf["length_scale"], inf["noise_w"]]))[0]})[0].squeeze()
    with wave.open(f"{EXP}/compare-{name}-compact-split.wav", "wb") as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(cfg["audio"]["sample_rate"])
        w.writeframes((np.clip(audio, -1, 1) * 32767).astype(np.int16).tobytes())

    # With randomness off, encoder -> decoder must give the same audio as the whole Compact file.
    a = full_s.run(None, feed([0.0, 1.0, 0.0]))[0].squeeze()
    b = dec_s.run(None, {"z": enc_s.run(None, feed([0.0, 1.0, 0.0]))[0]})[0].squeeze()
    diff = float(np.abs(a - b).max()) if a.shape == b.shape else float("inf")
    log(f"Split check: encoder+decoder vs whole Compact file, max difference {diff:.6f} (0 or tiny = good).")
    if diff > 1e-3:
        print("WARNING: the split files don't match the Compact file. Use the Compact .onnx + .onnx.json without the split pair.")
    log("Listen and compare in Dev/Dusk/export: compare-*-standard-fp32.wav, compare-*-compact-fp16.wav, compare-*-compact-split.wav")

In [ ]:
# Packages: dusk-standard.zip, dusk-compact-ios.zip, NVDA Sonata archive (.tar.gz), sherpa-onnx copy for Android/BrailleNote,
# a model card; then copy everything to Dev/Dusk/export.
import tarfile, onnx, urllib.request, zipfile
with tarfile.open(f"{EXP}/{VOICE_KEY}-nvda-sonata.tar.gz", "w:gz") as t:
    for f in (ONNX, JSON_):
        t.add(f, arcname=os.path.basename(f))

if EXPORT_SHERPA:
    S = f"{EXP}/vits-piper-{VOICE_KEY}"; os.makedirs(S, exist_ok=True)
    m = onnx.load(ONNX)                                               # same metadata k2-fsa's Piper converter adds
    for k, v in {"model_type": "vits", "comment": "piper", "language": "English", "voice": cfg["espeak"]["voice"],
                 "has_espeak": 1, "n_speakers": cfg["num_speakers"], "sample_rate": cfg["audio"]["sample_rate"]}.items():
        m.metadata_props.add(key=k, value=str(v))
    onnx.save(m, f"{S}/{VOICE_KEY}.onnx")
    with open(f"{S}/tokens.txt", "w", encoding="utf-8") as f:
        f.writelines(f"{s} {i[0]}\n" for s, i in cfg["phoneme_id_map"].items())
    if not os.path.isdir(f"{S}/espeak-ng-data"):
        urllib.request.urlretrieve("https://github.com/k2-fsa/sherpa-onnx/releases/download/tts-models/espeak-ng-data.tar.bz2",
                                   f"{LOCAL}/espeak-ng-data.tar.bz2")
        sh(f"tar -xjf {LOCAL}/espeak-ng-data.tar.bz2 -C {S}")
    sh(f"cd {EXP} && tar -cjf vits-piper-{VOICE_KEY}.tar.bz2 vits-piper-{VOICE_KEY}")

card = f"""# Model card for dusk (high)
* Language: en_US (English, United States)
* Speakers: 1
* Quality: high
* Samplerate: 22,050Hz

## Dataset
* Hi-Fi Multi-Speaker English TTS Dataset (Hi-Fi TTS), Bakhturina et al. (NVIDIA), 2021: https://www.openslr.org/109/
* License: CC BY 4.0. Reader {SPEAKER_ID} (LibriVox reader Celine Major); recordings from LibriVox (public domain).
* Changes: about {HOURS_OF_AUDIO} h of clips selected for pitch range, resampled to 22,050 Hz, loudness-normalized.

## Training
Fine-tuned with Piper (https://github.com/OHF-Voice/piper1-gpl) from the ljspeech high checkpoint by Bryce Beattie
(https://huggingface.co/datasets/rhasspy/piper-checkpoints), trained on the LJ Speech Dataset (public domain).
## Versions
* Dusk Standard: full-size (fp32) en_US-dusk-high.onnx + .onnx.json, for NVDA, Android, Piper on a computer.
* Dusk Compact: half-size (fp16) en_US-dusk-high.onnx + split .encoder.onnx / .decoder.onnx, for the iPhone app
  "Piper - Neural TTS". Same voice and settings file.

Co-written with Tab. Dusk is shared under CC BY 4.0.
"""
open(f"{EXP}/MODEL_CARD", "w").write(card)

# Dusk Standard: the full-size voice + settings + model card in one zip.
with zipfile.ZipFile(f"{EXP}/dusk-standard.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in (ONNX, JSON_, f"{EXP}/MODEL_CARD"):
        z.write(f, arcname=os.path.basename(f))

# Dusk Compact: half-size voice + split pair + settings + model card, in its own zip and its own Drive subfolder
# (same file names as Standard, so they never sit in the same folder).
if EXPORT_COMPACT:
    shutil.copy(f"{EXP}/MODEL_CARD", f"{COMPACT}/MODEL_CARD")
    with zipfile.ZipFile(f"{EXP}/dusk-compact-ios.zip", "w", zipfile.ZIP_STORED) as z:   # onnx barely compresses
        for f in (C_ONNX, C_JSON, C_ENC, C_DEC, f"{COMPACT}/MODEL_CARD"):
            z.write(f, arcname=os.path.basename(f))
    os.makedirs(f"{D}/export/compact-ios", exist_ok=True)
    for f in glob.glob(f"{COMPACT}/*"):
        shutil.copy(f, f"{D}/export/compact-ios/")
for f in glob.glob(f"{EXP}/*"):
    if os.path.isfile(f):
        shutil.copy(f, f"{D}/export/")
log("Done. Dusk/export has: " + ", ".join(sorted(os.listdir(f"{D}/export"))))

## If something goes wrong
- "CUDA out of memory": the training cell retries with batch 12, then 8. You can also set BATCH_SIZE lower in Settings.
- Losses show nan: set PRECISION back to "32".
- Colab says GPU usage limit reached: wait a day, then Run all. Nothing is lost; it resumes from Dev/Dusk/checkpoints.
- Want the voice early: set TRAIN = False and Run all. It exports the newest checkpoint.
- Listen to progress: Dev/Dusk/samples/epoch-XXXX. Dev/Dusk/samples/metrics.csv has words per minute and pitch spread (semitones) for each render.
- Drive getting full: replaced checkpoints (about 1 GB each) go to Google Drive's trash. The notebook never touches the trash or anything outside Dev/Dusk, so empty the trash yourself if you need the space.
- iPhone app won't take the split files: import just the Compact en_US-dusk-high.onnx + en_US-dusk-high.onnx.json from dusk-compact-ios.zip. Split voices are new in the app (test build first).